# The Slow & Fast Brains: Test-Time Compute for Physical AI, on Anyscale

### Scaling world-model *planning* in the cloud, then distilling it into a fast on-robot policy

Today's robot policies (VLAs) are dazzling **reflexes**: image + instruction → action, in one shot. But a reflex can't *deliberate*. It can't imagine three ways a grasp might fail and pick the one that won't. Large language models crossed exactly this threshold with **test-time compute**. This is the "think longer, answer better" unlock behind o1-style reasoning.

**This notebook plants a flag on the robotics version of that unlock**, and shows the production architecture for it on Ray + Anyscale:

> Give the robot a **world model** it can *imagine and search inside* to choose actions; run that search at scale in the cloud; and **distill** the result back into a fast policy that runs on the robot.

**Fast brain on the robot, slow brain in the cloud.**

## The bet

A world model predicts *what happens next* given an action. Once you have one, action selection becomes **search**: sample candidate action sequences, roll them forward in the model's imagination, and keep the one whose imagined outcome is closest to the goal. More compute → more candidates and longer lookahead → better actions. That is **test-time compute for control**.

Yann LeCun's team has bet heavily on this direction with **V-JEPA** — a self-supervised video world model that predicts in *latent* space rather than pixels. We use the open, action-conditioned **V-JEPA 2-AC** (MIT licensed) as our substrate.

The catch, which is the whole point of this template: **the search is expensive** — far too slow to run in a robot's millisecond control loop. So it belongs in the cloud, scaled horizontally, with the distilled policy deployed to the edge. That is precisely the workload Ray + Anyscale exist to run.

## The architecture

```
        ┌─────────────────────────  THE CLOUD (Anyscale)  ─────────────────────────────┐
        │                                                                              │
        │   V-JEPA 2-AC world model (1.3B)        Ray fans the search out:             │
        │   ┌────────────────────────┐            • DEEPER  (more compute / action)    │
        │   │  imagine futures + CEM │  × N GPUs  • WIDER   (more robots in parallel)  │
        │   │  search over actions   │                                                 │
        │   └────────────────────────┘            mines (state → best action) labels   │
        │              │                                      │                        │
        │              └────────────  distill  ───────────────┘                        │
        └──────────────────────────────│───────────────────────────────────────────────┘
                          periodic download │ of the fast policy
        ┌──────────────────────────────▼─────────────────────────────────────────────┐
        │   THE ROBOT:  fast distilled policy, milliseconds, no search               │
        └────────────────────────────────────────────────────────────────────────────┘
```

The slow brain thinks continuously in the cloud; the fleet pulls distilled updates. Let's build it.

---
## 1. Connect to the cluster

We ship our slow-brain module (`slow_brain.py`) to every node with `py_modules`, and point the world-model weights cache at shared cluster storage so each GPU loads the 11 GB checkpoint locally instead of re-downloading it.

In [ ]:
import os, sys, time, numpy as np, ray

REPO_DIR = "/home/ray/default/robotics_world_model_inference"
sys.path.insert(0, REPO_DIR)   # so the driver (this notebook) can import slow_brain too
TORCH_HOME = "/mnt/cluster_storage/torch_hub"   # shared cache: weights + V-JEPA repo
VJEPA_REPO = f"{TORCH_HOME}/hub/facebookresearch_vjepa2_main"

ray.init(address="auto", logging_level="ERROR", runtime_env={
    "py_modules": [f"{REPO_DIR}/slow_brain.py"],
    "env_vars": {"TORCH_HOME": TORCH_HOME, "HF_HOME": "/mnt/cluster_storage/hf_cache"},
})

n_gpus = int(ray.cluster_resources().get("GPU", 0))
print(f"Cluster GPUs available for slow-brain planners: {n_gpus}")

## 2. A task for the world model

We use a short Franka manipulation clip shipped with V-JEPA 2: a **context** frame (where the arm is now) and a **goal** frame (where we want it), plus the robot's proprioceptive **state** (7-DoF end-effector pose). The slow brain's job: find the action that moves the imagined latent from context toward goal.

In [ ]:
import matplotlib.pyplot as plt

traj = np.load(f"{VJEPA_REPO}/notebooks/franka_example_traj.npz")
ctx_img, goal_img = traj["observations"][0, 0], traj["observations"][0, 1]
pose = traj["states"][0, 0]   # 7-DoF end-effector pose

fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
ax[0].imshow(ctx_img); ax[0].set_title("context (now)"); ax[0].axis("off")
ax[1].imshow(goal_img); ax[1].set_title("goal"); ax[1].axis("off")
plt.tight_layout(); plt.show()
print("images:", ctx_img.shape, "| pose (7-DoF):", np.round(pose, 3))

## 3. The slow brain imagines and searches

We spin up one **`Planner`** — a Ray actor that loads V-JEPA 2-AC onto a GPU once and then serves planning calls. Inside, planning is the **Cross-Entropy Method (CEM)**: sample a batch of candidate actions, roll each one forward through the action-conditioned predictor *in latent space*, score how close the imagined result lands to the goal latent (lower **energy** = closer), keep the best, refit the sampling distribution, and repeat.

Two knobs set how hard it "thinks": **`samples`** (candidates per step) and **`cem_steps`** (refinement iterations). That product is our test-time compute dial.

In [ ]:
from slow_brain import Planner

planner = Planner.remote()   # loads the 1.3B world model onto one GPU (~45s, one time)

res = ray.get(planner.plan_from_images.remote(
    ctx_img, goal_img, pose, samples=64, cem_steps=4, rollout=1))

print(f"planned action (7-DoF delta): {np.round(res['action'], 3)}")
print(f"energy to goal (lower=better): {res['energy']:.4f}")
print(f"planning time on one L4:       {res['seconds']:.2f} s  (light config)")

## 4. Why this can't run on the robot

The cell above used a deliberately *light* config so it returns in seconds. A **production-strength** search — hundreds of candidates, ten refinement steps — is far heavier. On our dev cluster's **L4** GPU we measured it directly:

| config | samples × cem_steps × rollout | latency / action (L4) |
|---|---|---|
| light (above) | 64 × 4 × 1 | ~2.4 s |
| **production-strength** | **400 × 10 × 2** | **~98 s** |

A robot control loop runs at **10–50 ms**. At ~98 s/action, full-strength planning is **~2,000–10,000× too slow to run on-robot.** That gap is not a bug — it's the reason the slow brain lives in the cloud. Below we make the compute↔quality trade-off explicit, then scale it with Ray. *(Production hardware closes much of this gap — see §8.)*

---
## 5. Ray as the orchestration substrate

Planning scales along **two independent axes**, and Ray fans out both:

1. **Deeper** — spend more compute per action (bigger search) for a better action. This is the *o1 curve for robots*.
2. **Wider** — plan for many robots / many states at once. This is *fleet throughput*.

First, the **deeper** axis: sweep the compute dial and watch plan quality (energy) improve. We distribute the sweep across all available GPUs.

In [ ]:
# Reuse the planner from §3 and add one per *remaining* GPU, so we use exactly n_gpus GPUs.
# (Creating n_gpus NEW ones here would mean n_gpus+1 actors for n_gpus GPUs -> the extra deadlocks.)
planners = [planner] + [Planner.remote() for _ in range(max(0, n_gpus - 1))]
ray.get([p.plan_from_images.remote(ctx_img, goal_img, pose, samples=8, cem_steps=1)
         for p in planners])   # warmup / model load on every GPU
print(f"{len(planners)} slow-brain planners online (1 reused from §3).")

In [ ]:
# DEEPER axis: increasing test-time compute per action, fanned across GPUs.
compute_dial = [(32, 2), (64, 3), (128, 5), (256, 8)]   # (samples, cem_steps), rollout=1

futs = [planners[i % len(planners)].plan_from_images.remote(
            ctx_img, goal_img, pose, samples=s, cem_steps=c, rollout=1)
        for i, (s, c) in enumerate(compute_dial)]
sweep = ray.get(futs)

for (s, c), r in zip(compute_dial, sweep):
    print(f"samples={s:>3} cem_steps={c:>2}  (compute={s*c:>4})  "
          f"energy={r['energy']:.4f}  latency={r['seconds']:5.2f}s")

In [ ]:
# The 'o1 curve for robots': more test-time compute -> better plan (lower energy), at a latency cost.
compute = [s * c for s, c in compute_dial]
energy = [r["energy"] for r in sweep]
latency = [r["seconds"] for r in sweep]

fig, ax1 = plt.subplots(figsize=(6.5, 4))
ax1.plot(compute, energy, "o-", color="C0", lw=2, label="plan energy (↓ better)")
ax1.set_xlabel("test-time compute per action  (samples × CEM steps)")
ax1.set_ylabel("energy to goal", color="C0"); ax1.set_xscale("log")
ax2 = ax1.twinx()
ax2.plot(compute, latency, "s--", color="C3", lw=2, label="latency (s)")
ax2.set_ylabel("latency / action (s)", color="C3")
ax1.set_title("Test-time compute for robots: think longer, act better")
fig.tight_layout(); plt.show()

**Read the curve.** Spending more compute inside the world model buys better actions — the same "think longer, answer better" law that powers o1-style reasoning, now for control. The cloud is where you can afford to ride this curve; the robot is not. Ray lets you pick any point on it and run it at fleet scale.

## 6. The wider axis: fleet throughput

Now hold compute fixed and scale the **number of planners**. This is the real production question: *how many robot-states per second can the cloud plan for?* Ray scales it near-linearly with GPUs.

In [ ]:
# WIDER axis: fixed light config, measure throughput as we add planners (GPUs).
N_TASKS = 4 * len(planners)
cfg = dict(samples=64, cem_steps=4, rollout=1)

throughput = []
for k in range(1, len(planners) + 1):
    pool = planners[:k]
    t0 = time.time()
    ray.get([pool[i % k].plan_from_images.remote(ctx_img, goal_img, pose, **cfg)
             for i in range(N_TASKS)])
    dt = time.time() - t0
    throughput.append(N_TASKS / dt)
    print(f"{k} planner(s): {N_TASKS} actions in {dt:5.1f}s  ->  {N_TASKS/dt:.2f} actions/s")

plt.figure(figsize=(6, 3.8))
plt.plot(range(1, len(planners) + 1), throughput, "o-", lw=2)
plt.xlabel("# slow-brain planners (GPUs)"); plt.ylabel("actions planned / s")
plt.title("Fleet throughput scales with the cluster"); plt.tight_layout(); plt.show()

---
## 7. Closing the loop: distill the slow brain into a fast brain

Search is how you *discover* good actions; you don't want to pay for it forever. The **"o1 moment for robots"**: use the expensive planner as a **data engine** — mine `(state → best action)` pairs across the fleet — then **distill** them into a small policy that reproduces the planner's decisions in a single forward pass. The slow brain searches once; the fast brain inherits the answer and runs on-robot in milliseconds.

*(Illustrative scale: we mine a small label set from one task so the cell runs quickly. In production you mine continuously across a diverse task distribution — the exact workload Ray was built to sustain.)*

In [ ]:
# Slow brain as a data engine: mine (latent state -> planned action) labels across all GPUs.
# We jitter the observation per sample to create input variety for the demo.
N_LABELS = 4 * len(planners)
rng = np.random.default_rng(0)

def jitter(img):
    noisy = img.astype(np.int16) + rng.integers(-12, 12, img.shape)
    return np.clip(noisy, 0, 255).astype(np.uint8)

futs = [planners[i % len(planners)].plan_from_images.remote(
            jitter(ctx_img), goal_img, pose, samples=64, cem_steps=4, rollout=1)
        for i in range(N_LABELS)]
labels = ray.get(futs)
X = np.stack([r["feat"] for r in labels])      # mean-pooled world-model latents
Y = np.stack([r["action"] for r in labels])    # planner's chosen actions
print(f"mined {len(X)} labels  |  features {X.shape}  ->  actions {Y.shape}")

In [ ]:
# Train the fast brain: a small MLP that maps world-model features -> action, no search.
import torch, torch.nn as nn

Xt = torch.tensor(X, dtype=torch.float32); Yt = torch.tensor(Y, dtype=torch.float32)
Xt = (Xt - Xt.mean(0)) / (Xt.std(0) + 1e-6)
fast_policy = nn.Sequential(nn.Linear(Xt.shape[1], 256), nn.ReLU(),
                            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 7))
opt = torch.optim.Adam(fast_policy.parameters(), lr=1e-3)
for epoch in range(300):
    opt.zero_grad(); loss = ((fast_policy(Xt) - Yt) ** 2).mean(); loss.backward(); opt.step()
    if epoch % 100 == 0 or epoch == 299:
        print(f"epoch {epoch:>3}  imitation MSE = {loss.item():.5f}")

In [ ]:
# The payoff: the fast brain reproduces the planner's action in ~milliseconds, no search.
fast_policy.eval()
with torch.no_grad():
    t0 = time.time()
    for _ in range(1000):
        _ = fast_policy(Xt[:1])
    fast_ms = (time.time() - t0) / 1000 * 1e3

print(f"fast brain (distilled policy): {fast_ms:.3f} ms / action  — robot-deployable")
print(f"slow brain (full CEM search):  ~98,000 ms / action  — cloud-only")
print(f"speedup at inference: ~{98_000 / fast_ms:,.0f}x")

---
## 8. Scaling on production hardware

Every number above is from a **4× L4 dev cluster** — a modest inference GPU (~24 GB, ~120 bf16 TFLOPS). The L4 is what makes the gap so stark; production accelerators close most of it, and the Ray orchestration is **identical** — the curves simply shift down and to the left:

- **Faster GPUs.** H100 / H200 / B200 deliver **~10–30×** the bf16/fp8 throughput of an L4. The ~98 s full-strength plan drops toward **single-digit seconds per action**.
- **More VRAM → bigger search per forward.** 80–192 GB lets you run far larger CEM sample batches in a single forward pass — *better plans at equal wall-clock*, since CEM samples are just a batch dimension.
- **Data-parallel search.** A single plan's sample batch shards across GPUs with Ray — near-linear speedup *within one action*, on top of the across-action fan-out shown here.
- **Kernel-level wins.** fp8 (Hopper/Blackwell), `torch.compile`, and FlashAttention compound for several more multiples.
- **The fast brain is unaffected** — it already runs in milliseconds and ships to the robot regardless of cloud hardware.

The thesis is hardware-agnostic: **planning is elastic cloud compute, deployment is a cheap distilled policy.** Better hardware buys deeper search or more robots per dollar — not a different architecture.

## 9. The standing architecture

Run this loop **continuously**: the slow brain plans across the fleet's hardest states in the cloud, mines fresh `(state → action)` labels, and distills them into the fast policy; robots pull periodic policy downloads. The fleet's reflexes keep improving from collective experience — a **flywheel** where compute spent thinking in the cloud becomes capability deployed at the edge.

That is what production-scale Physical AI looks like, and it is a Ray + Anyscale workload end to end: stateful GPU actors, elastic fan-out, shared storage, and a distillation pipeline — orchestrated as one system.

### What we showed

1. A world model (**V-JEPA 2-AC**) the robot can imagine and **search** inside — test-time compute for control.
2. The **compute↔quality curve** (deeper) and **fleet throughput** (wider), both fanned out with Ray.
3. Why search is **cloud-only** (~98 s/action on an L4) — and how production hardware shrinks that.
4. **Distillation** of the slow brain into a millisecond fast brain — the o1 moment for robots.

**References:** V-JEPA 2 (Meta AI, 2024–2025, `facebookresearch/vjepa2`, MIT) · Cross-Entropy Method for planning · o1-style test-time compute.

In [ ]:
# Release the GPU actors (the §3 planner is included in `planners`).
for p in planners:
    ray.kill(p)
print("planners released.")